# Exponenttruncatie: bronschaal, schaal ×64 en afrondingscontrole

Deze notebook presenteert één vooraf vastgelegde **post-hocevaluatie** van twee bestaande 100-epochtrainingen:

- **Direct getraind:** de drie MX-gewichtsprecisies werden tijdens de training rechtstreeks uit de bron bepaald.
- **Met naïeve truncatie getraind:** E2M3 ontstond via exponenttruncatie uit E4M3; E2M1 daarna via mantissatruncatie, met behoud van de E4M3-bronschaal.

Elke training houdt haar eigen **epoch-100-checkpoint** en haar reeds opgeslagen **E4M3-best-checkpoint**. De notebook toont die vier groepen afzonderlijk. Zij kiest geen nieuw checkpoint, route of schaal. De drie oorspronkelijke gewichtmodi worden opnieuw geëvalueerd als referentie; daarnaast worden zes A/B/C-routes uit dezelfde opgeslagen E4M3-bron afgeleid.

**A32, de opgeslagen BN-bank van de doelprecisie en alle overige modelparameters blijven vast.** De evaluator gebruikt de opgeslagen MX-export, zonder de zes FP32-gewichtmasters te openen. Deze notebook leest uitsluitend JSON, CSV en NPZ; zij voert zelf geen inferentie of training uit.

## 1. Resultaatmap

Open bij voorkeur de kopie in de nieuwe evaluatiemap. Vul anders hieronder het volledige pad naar die map in. Dit is de map met `run_report.json`, `metrics.csv` en de submappen `direct` en `truncation`.

`SAVE_FIGURES = False` toont alle figuren zonder bestanden te schrijven. Zet dit alleen op `True` wanneer je PNG- en PDF-versies wilt bewaren; dan ontstaat een nieuwe analysemap en blijven de ruwe resultaten behouden. Er wordt geen uitgevoerde notebook opgeslagen.

In [ ]:
from pathlib import Path

RUN_DIR = Path.cwd()  # Vervang zo nodig door Path(r"C:\Users\renzo\master-thesis-mx\mx_runs\mx_triple_scale64_posthoc_eval\JOUW_EVALUATIEMAP")
SAVE_FIGURES = False

In [ ]:
import importlib.util
import sys
import pandas as pd
from IPython.display import display, Markdown

helper_path = RUN_DIR.resolve() / "notebook_support.py"
if not helper_path.is_file():
    helper_path = Path.cwd() / "notebook_support.py"
if not helper_path.is_file():
    raise FileNotFoundError("Plaats deze notebook en notebook_support.py samen in de evaluatiemap en stel RUN_DIR in.")
spec = importlib.util.spec_from_file_location("mx_scale64_notebook_support", helper_path)
ns = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = ns
spec.loader.exec_module(ns)

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 16)
pd.set_option("display.max_colwidth", 70)
run = ns.load_run(RUN_DIR, verify_per_image=True)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.diagnostics_table(run))

## 2. Wat vergelijken we?

Noem de E4M3-blokschaal **S** en de gedecodeerde lokale E4M3-waarde **u**; het opgeslagen brongewicht is dus **S × u**.

| Familie | E2M3-bewerking | Blokschaal | E2M1-bewerking |
|---|---|---|---|
| A | Behoud teken, hoogste twee van vier exponentbits en alle drie mantissabits | S | Verwijder de laagste twee mantissabits van A6 |
| B | Exact dezelfde elementcodes als A6 | 64S | Dezelfde mantissatruncatie vanuit B6 |
| C | Rond **u / 64** af naar E2M3, nearest-even | 64S | Dezelfde mantissatruncatie vanuit C6 |

**A → B** verandert alleen de schaal; elk gedecodeerd B-gewicht is 64 keer zijn A-tegenhanger. **B → C** vergelijkt twee codeconstructies bij dezelfde vooraf gekozen schaal 64S. C hergebruikt uitsluitend de opgeslagen E4M3-informatie. Er wordt geen nieuwe per-blokschaal bepaald.

Deze controle test of gewone afronding vanuit diezelfde bron bij 64S beter werkt dan exponent-MSB-behoud. C is geen bovengrens op de netwerkaccuracy. De vaste BN-statistieken kunnen op een gewichtsverandering reageren; de twee accuracyverschillen vormen geen onafhankelijke, optelbare causale verklaring. De mantissatruncatie blijft in alle vierbitsroutes aanwezig.

In [ ]:
display(ns.route_table())

## 3. Hoofdresultaat: accuracy op beide vaste checkpoints

Elke training krijgt een eigen figuur met **epoch 100** links en **E4M3-best** rechts. Ook wanneer een lagere precisie een hogere accuracy haalt, blijft de eerder gekozen E4M3-best-rol ongewijzigd. De referentiebalken zijn de opnieuw gemeten oorspronkelijke modi van de desbetreffende training; bij de truncatietraining kunnen zij daarom samenvallen met A.

De tabellen tonen naast percentages ook de aantallen correcte voorspellingen en hard cross-entropy. Bij ontbrekende of inconsistente metriekgegevens wordt geen volledige accuracyfiguur ingevuld. Controleer altijd eerst de status bovenaan.

In [ ]:
display(ns.accuracy_table(run))
figures["accuracy_direct"] = ns.plot_checkpoint_accuracy(run, "direct")
display(figures["accuracy_direct"])
figures["accuracy_truncation"] = ns.plot_checkpoint_accuracy(run, "truncation")
display(figures["accuracy_truncation"])

## 4. Afzonderlijke verschillen: schaal en afronding

Een positieve balk betekent een hogere accuracy bij de nieuwe route. **B − A** laat de verandering door schaal ×64 zien bij gelijke codes. **C − B** laat de verandering door elementafronding zien bij gelijke schaal 64S. De volledige deltatafel bevat ook de verschillen met de oorspronkelijke getrainde doelmodus en C − A.

Vergelijk eerst binnen dezelfde training en checkpointrol. Verschillen tussen de twee trainingspopulaties zijn beschrijvend; deze proef bevat geen herhaalde trainingsseeds of onzekerheidsmeting over trainingen.

In [ ]:
display(ns.delta_table(run))
figures["accuracy_deltas"] = ns.plot_accuracy_deltas(run)
display(figures["accuracy_deltas"])

## 5. Welke voorspellingen veranderen?

De figuur splitst elke vergelijking op in beelden die van fout naar correct gaan en beelden die van correct naar fout gaan. Het verschil tussen die twee aandelen is de accuracydelta. Ook veranderingen van het ene verkeerde cijfer naar een ander verkeerd cijfer worden in de tabel bijgehouden.

De helper controleert de opgeslagen per-beeldvoorspellingen en logits, de aantallen, de gemiddelde cross-entropy en alle gepaarde vergelijkingen opnieuw. Gelijke maximale logits zijn toegestaan: de door de evaluator opgeslagen topk-keuze hoeft bij een tie niet gelijk te zijn aan NumPy-argmax.

In [ ]:
display(pd.DataFrame(run.output_checks))
if not run.pairs.empty:
    columns = [c for c in ["training", "role", "epoch", "reference_path", "other_path", "prediction_changes", "wrong_to_correct", "correct_to_wrong", "wrong_to_other_wrong", "net_correct_change", "accuracy_difference_pp"] if c in run.pairs]
    display(run.pairs[columns])
figures["paired_changes"] = ns.plot_paired_changes(run)
display(figures["paired_changes"])

## 6. Gewichtsgrootte en numerieke afwijking

Beide figuren gebruiken **dezelfde opgeslagen E4M3-bron** als gemeenschappelijke referentie per checkpoint. RMS laat de gewichtsgrootte zien; relatieve L2-afwijking is `||W_route − W_E4M3||₂ / ||W_E4M3||₂` per MX-laag. Een kleinere numerieke afwijking garandeert geen hogere accuracy. De FP32-master is geen referentie in deze proef.

Volle lijnen horen bij E2M3; stippellijnen bij E2M1. De laaglabels verwijzen naar de zes bestaande MX-convolutielagen. Bij een nulnorm kan een relatieve afwijking ongedefinieerd zijn; die wordt niet als nul ingevuld.

De samenvatting toont het bereik van de RMS-verhouding over de zes lagen en het totale aantal nulgewichten. A gebruikt S; B en C gebruiken doelbewust 64S. In de ruwe laagdiagnostiek verwijzen de historische veldnamen `*_to_direct` naar de **opgeslagen getrainde doelmodus van dezelfde training**, ook bij de truncatietraining. Cross-formaatcodebytes zijn geen vergelijkbare numerieke foutmaat.

In [ ]:
if not run.codes.empty and {"training", "role", "path", "rms_ratio_to_source", "zero_count", "n_weights", "scale_exponent_offset"} <= set(run.codes):
    code_summary = run.codes.groupby(["training", "role", "path"], sort=False).agg(
        rms_ratio_min=("rms_ratio_to_source", "min"), rms_ratio_max=("rms_ratio_to_source", "max"),
        nulgewichten=("zero_count", "sum"), aantal_gewichten=("n_weights", "sum"),
        schaaloffset=("scale_exponent_offset", "first")).reset_index()
    display(code_summary)
figures["weight_magnitudes"] = ns.plot_weight_magnitudes(run)
display(figures["weight_magnitudes"])
figures["weight_differences"] = ns.plot_weight_differences(run)
display(figures["weight_differences"])

## 7. Concentratie op één voorspelde klasse

Een lage accuracy kan samengaan met een vrijwel constante voorspelling. Daarom tonen we per route het aandeel van het meest voorspelde cijfer en het aantal verschillende voorspelde klassen. Eén klasse met 100% concentratie is een geldig gemeten resultaat; het is geen reden om die route uit de vergelijking te verwijderen.

Deze diagnose vervangt de accuracy niet. Zij helpt verklaren of een route nog gevarieerde voorspellingen produceert en of schaal ×64 of afronding dat patroon verandert.

In [ ]:
prediction_summary = ns.prediction_table(run)
if not prediction_summary.empty:
    display(prediction_summary[["training", "role", "epoch", "path", "classes_predicted", "dominant_class", "dominant_fraction", "n"]])
figures["prediction_diversity"] = ns.plot_prediction_diversity(run)
display(figures["prediction_diversity"])

## 8. Herhaling, vaste selectie en interpretatie

De historische herhalingscontrole vergelijkt de drie oorspronkelijke modi met de opgeslagen trainingresultaten. Een verschil wordt zichtbaar gerapporteerd. Gebruik bij de A/B/C-vergelijking altijd de huidige meting op dezelfde testbeelden; verwar een eventuele herhalingsafwijking niet met een truncatie-effect.

De audittabel bewaart per training en checkpoint de oorspronkelijke identiteit en bevestigt de vaste model- en BN-toestand. Er zijn nul optimizerupdates. De testset wordt eenmaal in dezelfde vaste volgorde per route verwerkt; train- en extra-data worden niet gebruikt.

Bespreek de uitkomst in deze volgorde: (1) reproduceert de oorspronkelijke modus, (2) wat doet A → B, (3) wat doet B → C, (4) komt dat overeen met gewichts- en voorspeldiagnostiek, en (5) verschillen die patronen tussen de twee trainingen? Een eventuele volgende trainingsproef volgt pas na beoordeling van deze resultaten.

In [ ]:
display(run.reproduction)
display(ns.audit_table(run))
display(Markdown(ns.timing_markdown(run)))
for limitation in run.report.get("limitations", []):
    display(Markdown("- " + str(limitation)))

## 9. Figuren bewaren

Alleen de volgende cel schrijft bestanden wanneer `SAVE_FIGURES` bovenaan op `True` staat. De nieuwe map bevat zeven PNG-figuren, zeven PDF-figuren en herkomstinformatie. Bestaande resultaatbestanden en notebooks worden behouden.

In [ ]:
if SAVE_FIGURES:
    saved_to = ns.save_figures(run, figures)
    print("Figuren bewaard in:", saved_to)
else:
    print("Figuren alleen getoond; geen analysebestanden geschreven.")